In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Smart Retail Inventory") \
    .getOrCreate()

In [2]:
stores_df = spark.read.csv("stores.csv", header=True, inferSchema=True)

products_df = spark.read.csv("products.csv", header=True, inferSchema=True)

inventory_df = spark.read.csv("inventory.csv", header=True, inferSchema=True)

sales_df = spark.read.csv("sales.csv", header=True, inferSchema=True)

suppliers_df = spark.read.option("multiline", True).json("suppliers.json")

In [3]:
stores_df.printSchema()
products_df.printSchema()
inventory_df.printSchema()
sales_df.printSchema()
suppliers_df.printSchema()

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: int

In [4]:
print(stores_df.count())
print(products_df.count())
print(inventory_df.count())
print(sales_df.count())
print(suppliers_df.count())

8
12
12
15
5


In [5]:
stores_df.write.mode("overwrite").parquet("bronze/stores")
products_df.write.mode("overwrite").parquet("bronze/products")
inventory_df.write.mode("overwrite").parquet("bronze/inventory")
sales_df.write.mode("overwrite").parquet("bronze/sales")
suppliers_df.write.mode("overwrite").parquet("bronze/suppliers")

In [7]:
products_df.filter(col("supplier_id").isNull()).show()

inventory_df.filter(col("stock_quantity").isNull()).show()

sales_df.filter(col("sale_amount").isNull()).show()

sales_df.filter(col("payment_m").isNull()).show()

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+

+-------+--------+----------+----------+-------------+-----------+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|
+-------+--------+----------+----------+-------------+-----------+---------+
| SA1011|    S102|      P103|2026-01-17|            1|       NULL| 

In [9]:
inventory_clean = inventory_df.fillna({"stock_quantity":0})

sales_clean = sales_df.fillna({
    "sale_amount":0,
    "payment_m":"Not Provided"
})

products_clean = products_df.fillna({
    "supplier_id":"UNKNOWN"
})

In [10]:
products_clean = products_clean.withColumn(
    "data_quality_status",
    when(col("supplier_id")=="UNKNOWN","Invalid")
    .otherwise("Valid")
)

inventory_clean = inventory_clean.withColumn(
    "data_quality_status",
    when(col("stock_quantity")==0,"Missing Stock")
    .otherwise("Valid")
)

sales_clean = sales_clean.withColumn(
    "data_quality_status",
    when(col("sale_amount")==0,"Missing Amount")
    .otherwise("Valid")
)

In [11]:
products_clean.write.mode("overwrite").parquet("silver/products")
inventory_clean.write.mode("overwrite").parquet("silver/inventory")
sales_clean.write.mode("overwrite").parquet("silver/sales")

In [12]:
suppliers_flat = suppliers_df.select(
"supplier_id",
"supplier_name",
"city",
"rating",
col("contact.phone").alias("phone"),
col("contact.email").alias("email")
)

In [13]:
col("contact.phone").alias("phone")

Column<'contact.phone AS phone'>

In [14]:
col("contact.email").alias("email")

Column<'contact.email AS email'>

In [16]:
suppliers_flat = suppliers_flat.fillna({"phone":"Not Provided"})
suppliers_flat.show()

+-----------+--------------------+---------+------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|       phone|               email|
+-----------+--------------------+---------+------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|  9876500013|                NULL|
|       S204|  Urban Furniture Co|    Delhi|   4.0|  9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|Not Provided|                NULL|
+-----------+--------------------+---------+------+------------+--------------------+



In [17]:
suppliers_flat = suppliers_flat.fillna({"email":"Not Provided"})
suppliers_flat.show()

+-----------+--------------------+---------+------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|       phone|               email|
+-----------+--------------------+---------+------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|  9876500013|        Not Provided|
|       S204|  Urban Furniture Co|    Delhi|   4.0|  9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|Not Provided|        Not Provided|
+-----------+--------------------+---------+------+------------+--------------------+



In [18]:
suppliers_flat.write.mode("overwrite").parquet("silver/suppliers")

In [19]:
product_supplier = products_clean.join(
suppliers_flat,
"supplier_id",
"left")
product_supplier.show()

+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|       supplier_name|     city|rating|       phone|               email|
+-----------+----------+------------+-----------+------------+----------+-------------------+--------------------+---------+------+------------+--------------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|              Valid|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|              Valid|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|
|       S203|      P103|  Television|Electronics|          LG|     45000|              Valid|     HomeTech Supply|   Mumbai|   4.4|  9876500013|        Not Provided|
|   

In [20]:
inventory_product = inventory_clean.join(
products_clean,
"product_id",
"left")
inventory_product.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P104|       I1003|    S101|             3|            5| 2026-01-11|              Valid|Office Chair|  Furni

In [21]:
sales_store = sales_clean.join(
stores_df,
"store_id",
"left")
sales_store.show()

+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|store_id|sale_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|          store_name|     city|      state| store_type|manager_name|
+--------+-------+----------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+
|    S101| SA1001|      P101|2026-01-10|            1|      65000|         UPI|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S101| SA1002|      P102|2026-01-10|            2|      50000|        Card|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|    S102| SA1003|      P101|2026-01-11|            1|      65000|         UPI|              Valid|Metro Mart Bangalore|Bangalore|  Karnataka|Supermarket| Priya

In [22]:
sales_product = sales_clean.join(
products_clean,
"product_id",
"left")
sales_product.show()

+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101| SA1001|    S101|2026-01-10|            1|      65000|         UPI|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|      P102| SA1002|    S101|2026-01-10|            2|      50000|        Card|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|
|      P101| SA1003|    S102|2026-01-11|            1|      65000|         UPI|              Va

In [23]:
retail_df = sales_clean.join(stores_df,"store_id","left")\
                       .join(products_clean,"product_id","left")
retail_df.show()

+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|store_id|sale_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|          store_name|     city|      state| store_type|manager_name|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+--------+-------+----------+-------------+-----------+------------+-------------------+--------------------+---------+-----------+-----------+------------+------------+-----------+------------+-----------+----------+-------------------+
|      P101|    S101| SA1001|2026-01-10|            1|      65000|         UPI|              Valid|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|
|   

In [24]:
products_clean.join(
suppliers_flat,
"supplier_id",
"leftanti").show()

+-----------+----------+------------+-----------+---------+----------+-------------------+
|supplier_id|product_id|product_name|   category|    brand|unit_price|data_quality_status|
+-----------+----------+------------+-----------+---------+----------+-------------------+
|       S206|      P107|       Watch|    Fashion| Fastrack|      8000|              Valid|
|       S206|      P108|    Backpack|    Fashion|Wildcraft|      2500|              Valid|
|       S999|      P111|  Headphones|Electronics|     Sony|      3000|              Valid|
|    UNKNOWN|      P112|     T-Shirt|    Fashion|     Puma|      1500|            Invalid|
+-----------+----------+------------+-----------+---------+----------+-------------------+



In [25]:
inventory_clean.join(
products_clean,
"product_id",
"leftanti").show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+
|      P120|       I1012|    S108|            12|            5| 2026-01-18|              Valid|
+----------+------------+--------+--------------+-------------+-----------+-------------------+



In [26]:
sales_clean.join(
products_clean,
"product_id",
"leftanti").show()

+----------+-------+--------+----------+-------------+-----------+---------+-------------------+
|product_id|sale_id|store_id| sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|
+----------+-------+--------+----------+-------------+-----------+---------+-------------------+
|      P120| SA1009|    S108|2026-01-15|            2|      10000|     Cash|              Valid|
+----------+-------+--------+----------+-------------+-----------+---------+-------------------+



In [27]:
sales_clean.join(
stores_df,
"store_id",
"leftanti").show()

+--------+-------+----------+---------+-------------+-----------+---------+-------------------+
|store_id|sale_id|product_id|sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|
+--------+-------+----------+---------+-------------+-----------+---------+-------------------+
+--------+-------+----------+---------+-------------+-----------+---------+-------------------+



In [29]:
inventory_product = inventory_product.withColumn(
"stock_status",
when(col("stock_quantity")<=col("reorder_level"),
"Reorder Required")
.otherwise("Sufficient Stock"))
inventory_product.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|    stock_status|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|Sufficient Stock|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|Sufficient Stock|
|      P104|       I1003|    S

In [30]:
products_clean = products_clean.withColumn(
"price_category",
when(col("unit_price")>=50000,"Premium")
.when(col("unit_price")>=10000,"Standard")
.otherwise("Budget"))
products_clean.show()

+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+------------+-----------+----------+-------------------+--------------+
|      P101|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|       Premium|
|      P102|      Mobile|Electronics|     Samsung|       S202|     25000|              Valid|      Standard|
|      P103|  Television|Electronics|          LG|       S203|     45000|              Valid|      Standard|
|      P104|Office Chair|  Furniture| Featherlite|       S204|      7000|              Valid|        Budget|
|      P105| Study Table|  Furniture|Urban Ladder|       S204|     12000|              Valid|      Standard|
|      P106|       Shoes|    Fashion|        Nike|       S205|      4500|              Valid|        Budget|
|      P107|       

In [31]:
sales_clean = sales_clean.withColumn(
"revenue_category",
when(col("sale_amount")>=50000,"High Revenue")
.when(col("sale_amount")>=15000,"Medium Revenue")
.otherwise("Low Revenue"))
sales_clean.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     Low Revenue|
| SA1006|    S105|      P108|2026-01-13|            5|      1250

In [32]:
sales_clean = sales_clean.withColumn(
"month",month("sale_date"))
sales_clean.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     Low Revenue|    1|
| SA1006|    S10

In [33]:
sales_clean = sales_clean.withColumn(
"year",year("sale_date"))
sales_clean.show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     

In [34]:
inventory_product = inventory_product.withColumn(
"inventory_value",
col("stock_quantity")*col("unit_price"))
inventory_product.show()

+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+---------------+
|product_id|inventory_id|store_id|stock_quantity|reorder_level|last_update|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|    stock_status|inventory_value|
+----------+------------+--------+--------------+-------------+-----------+-------------------+------------+-----------+------------+-----------+----------+-------------------+----------------+---------------+
|      P101|       I1001|    S101|            10|            5| 2026-01-10|              Valid|      Laptop|Electronics|      Lenovo|       S201|     65000|              Valid|Sufficient Stock|         650000|
|      P102|       I1002|    S101|            25|           10| 2026-01-10|              Valid|      Mobile|Electronics|     Samsung|       S202|     25000|    

In [35]:
suppliers_flat = suppliers_flat.withColumn(
"supplier_quality",
when(col("rating")>=4.5,"Excellent")
.when(col("rating")>=4.0,"Good")
.otherwise("Average"))
suppliers_flat.show()

+-----------+--------------------+---------+------+------------+--------------------+----------------+
|supplier_id|       supplier_name|     city|rating|       phone|               email|supplier_quality|
+-----------+--------------------+---------+------+------------+--------------------+----------------+
|       S201|    TechSource India|Hyderabad|   4.5|  9876500011| techsource@mail.com|       Excellent|
|       S202|MobileWorld Distr...|Bangalore|   4.2|Not Provided|mobileworld@mail.com|            Good|
|       S203|     HomeTech Supply|   Mumbai|   4.4|  9876500013|        Not Provided|            Good|
|       S204|  Urban Furniture Co|    Delhi|   4.0|  9876500014|      urban@mail.com|            Good|
|       S205|      Fashion Direct|     Pune|   3.8|Not Provided|        Not Provided|         Average|
+-----------+--------------------+---------+------+------------+--------------------+----------------+



In [36]:
stores_df.groupBy("state").count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    1|
|     Kerala|    1|
| Tamil Nadu|    1|
|      Delhi|    1|
|  Rajasthan|    1|
|  Telangana|    1|
|Maharashtra|    2|
+-----------+-----+



In [37]:
products_clean.groupBy("category").count().show()

+-----------+-----+
|   category|count|
+-----------+-----+
|    Fashion|    4|
|Electronics|    5|
|  Furniture|    3|
+-----------+-----+



In [38]:
products_clean.groupBy("brand").count().show()

+------------+-----+
|       brand|count|
+------------+-----+
|        Nike|    1|
|        Sony|    1|
|Urban Ladder|    1|
|        Puma|    1|
|      Lenovo|    1|
| Featherlite|    1|
|     Samsung|    1|
|      Godrej|    1|
|          LG|    1|
|   Wildcraft|    1|
|    Fastrack|    1|
|   Whirlpool|    1|
+------------+-----+



In [39]:
inventory_product.groupBy("store_id")\
.sum("inventory_value").show()

+--------+--------------------+
|store_id|sum(inventory_value)|
+--------+--------------------+
|    S105|              125000|
|    S102|              745000|
|    S106|                   0|
|    S104|               32000|
|    S107|               32000|
|    S101|             1296000|
|    S108|                NULL|
|    S103|              159000|
+--------+--------------------+



In [40]:
inventory_product.groupBy("category")\
.sum("inventory_value").show()

+-----------+--------------------+
|   category|sum(inventory_value)|
+-----------+--------------------+
|    Fashion|              292000|
|       NULL|                NULL|
|Electronics|             2020000|
|  Furniture|               77000|
+-----------+--------------------+



In [41]:
inventory_product.filter(
col("stock_status")=="Reorder Required").count()

5

In [42]:
sales_clean.agg(sum("sale_amount")).show()

+----------------+
|sum(sale_amount)|
+----------------+
|          373000|
+----------------+



In [43]:
retail_df.groupBy("store_name")\
.sum("sale_amount").show()

+--------------------+----------------+
|          store_name|sum(sale_amount)|
+--------------------+----------------+
|Metro Mart Bangalore|           65000|
|    Metro Mart Kochi|           32000|
|   Metro Mart Jaipur|           10000|
|   Metro Mart Mumbai|           30000|
|     Metro Mart Pune|           38000|
|Metro Mart Hyderabad|          154000|
|    Metro Mart Delhi|           20000|
|  Metro Mart Chennai|           24000|
+--------------------+----------------+



In [44]:
retail_df.groupBy("city")\
.sum("sale_amount").show()

+---------+----------------+
|     city|sum(sale_amount)|
+---------+----------------+
|Bangalore|           65000|
|    Kochi|           32000|
|  Chennai|           24000|
|   Mumbai|           30000|
|     Pune|           38000|
|    Delhi|           20000|
|Hyderabad|          154000|
|   Jaipur|           10000|
+---------+----------------+



In [45]:
retail_df.groupBy("category")\
.sum("sale_amount").show()

+-----------+----------------+
|   category|sum(sale_amount)|
+-----------+----------------+
|    Fashion|           62000|
|       NULL|           10000|
|Electronics|          243000|
|  Furniture|           58000|
+-----------+----------------+



In [46]:
retail_df.groupBy("product_name") \
    .agg(sum("sale_amount").alias("total_revenue")) \
    .show()

+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|Office Chair|        14000|
|        NULL|        10000|
|Refrigerator|        38000|
|      Laptop|       130000|
|        Sofa|        32000|
|    Backpack|        20000|
|       Shoes|        18000|
|      Mobile|        75000|
|  Television|            0|
| Study Table|        12000|
|       Watch|        24000|
+------------+-------------+



In [48]:
sales_clean.groupBy("payment_m") \
    .agg(sum("sale_amount").alias("total_revenue")) \
    .show()

+------------+-------------+
|   payment_m|total_revenue|
+------------+-------------+
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
|         UPI|       190500|
+------------+-------------+



In [49]:
retail_df.groupBy("product_name") \
    .agg(sum("sale_amount").alias("revenue")) \
    .orderBy(desc("revenue")) \
    .show(1)

+------------+-------+
|product_name|revenue|
+------------+-------+
|      Laptop| 130000|
+------------+-------+
only showing top 1 row


In [50]:
retail_df.groupBy("store_name") \
    .agg(sum("sale_amount").alias("revenue")) \
    .orderBy(desc("revenue")) \
    .show(1)

+--------------------+-------+
|          store_name|revenue|
+--------------------+-------+
|Metro Mart Hyderabad| 154000|
+--------------------+-------+
only showing top 1 row


In [51]:
retail_df.groupBy("category") \
    .agg(sum("sale_amount").alias("revenue")) \
    .orderBy(desc("revenue")) \
    .show(1)

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 243000|
+-----------+-------+
only showing top 1 row


In [52]:
product_revenue = retail_df.groupBy("product_name") \
    .agg(sum("sale_amount").alias("revenue"))

windowSpec = Window.orderBy(desc("revenue"))

product_revenue.withColumn(
    "rank",
    rank().over(windowSpec)
).show()

+------------+-------+----+
|product_name|revenue|rank|
+------------+-------+----+
|      Laptop| 130000|   1|
|      Mobile|  75000|   2|
|Refrigerator|  38000|   3|
|        Sofa|  32000|   4|
|       Watch|  24000|   5|
|    Backpack|  20000|   6|
|       Shoes|  18000|   7|
|Office Chair|  14000|   8|
| Study Table|  12000|   9|
|        NULL|  10000|  10|
|  Television|      0|  11|
+------------+-------+----+



In [54]:
store_revenue = retail_df.groupBy("store_name") \
    .agg(sum("sale_amount").alias("revenue"))

store_revenue.withColumn(
    "rank",
    rank().over(Window.orderBy(desc("revenue")))
).show()

+--------------------+-------+----+
|          store_name|revenue|rank|
+--------------------+-------+----+
|Metro Mart Hyderabad| 154000|   1|
|Metro Mart Bangalore|  65000|   2|
|     Metro Mart Pune|  38000|   3|
|    Metro Mart Kochi|  32000|   4|
|   Metro Mart Mumbai|  30000|   5|
|  Metro Mart Chennai|  24000|   6|
|    Metro Mart Delhi|  20000|   7|
|   Metro Mart Jaipur|  10000|   8|
+--------------------+-------+----+



In [55]:
category_revenue = retail_df.groupBy(
    "category","product_name"
).agg(sum("sale_amount").alias("revenue"))

windowSpec = Window.partitionBy("category") \
                   .orderBy(desc("revenue"))

category_revenue.withColumn(
    "rank",
    rank().over(windowSpec)
).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|Electronics|      Mobile|  75000|   2|
|Electronics|Refrigerator|  38000|   3|
|Electronics|  Television|      0|   4|
|    Fashion|       Watch|  24000|   1|
|    Fashion|    Backpack|  20000|   2|
|    Fashion|       Shoes|  18000|   3|
|  Furniture|        Sofa|  32000|   1|
|  Furniture|Office Chair|  14000|   2|
|  Furniture| Study Table|  12000|   3|
+-----------+------------+-------+----+



In [56]:
category_revenue.withColumn(
    "rank",
    rank().over(windowSpec)
).filter(
    col("rank")==1
).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|    Fashion|       Watch|  24000|   1|
|  Furniture|        Sofa|  32000|   1|
+-----------+------------+-------+----+



In [57]:
category_revenue.withColumn(
    "rank",
    rank().over(windowSpec)
).filter(
    col("rank")<=3
).show()

+-----------+------------+-------+----+
|   category|product_name|revenue|rank|
+-----------+------------+-------+----+
|       NULL|        NULL|  10000|   1|
|Electronics|      Laptop| 130000|   1|
|Electronics|      Mobile|  75000|   2|
|Electronics|Refrigerator|  38000|   3|
|    Fashion|       Watch|  24000|   1|
|    Fashion|    Backpack|  20000|   2|
|    Fashion|       Shoes|  18000|   3|
|  Furniture|        Sofa|  32000|   1|
|  Furniture|Office Chair|  14000|   2|
|  Furniture| Study Table|  12000|   3|
+-----------+------------+-------+----+



In [58]:
store_state = retail_df.groupBy(
    "state","store_name"
).agg(sum("sale_amount").alias("revenue"))

windowSpec = Window.partitionBy("state") \
                   .orderBy(desc("revenue"))

store_state.withColumn(
    "rank",
    rank().over(windowSpec)
).filter(
    col("rank")==1
).show()

+-----------+--------------------+-------+----+
|      state|          store_name|revenue|rank|
+-----------+--------------------+-------+----+
|      Delhi|    Metro Mart Delhi|  20000|   1|
|  Karnataka|Metro Mart Bangalore|  65000|   1|
|     Kerala|    Metro Mart Kochi|  32000|   1|
|Maharashtra|     Metro Mart Pune|  38000|   1|
|  Rajasthan|   Metro Mart Jaipur|  10000|   1|
| Tamil Nadu|  Metro Mart Chennai|  24000|   1|
|  Telangana|Metro Mart Hyderabad| 154000|   1|
+-----------+--------------------+-------+----+



In [59]:
windowSpec = Window.orderBy("sale_date")

sales_clean.withColumn(
    "running_total",
    sum("sale_amount").over(windowSpec)
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|year|running_total|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|       115000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|       115000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|       180000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|       206000|
| SA10

In [60]:
sales_clean.withColumn(
    "previous_sale",
    lag("sale_amount",1).over(
        Window.orderBy("sale_date"))
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|year|previous_sale|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+-------------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|         NULL|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|        65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|        50000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|        65000|
| SA10

In [61]:
sales_clean.withColumn(
    "next_sale",
    lead("sale_amount",1).over(
        Window.orderBy("sale_date"))
).show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|year|next_sale|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+---------+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|    50000|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|    65000|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|    18000|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|     8000|
| SA1005|    S104|      P107|2026-

In [62]:
sales_product = retail_df.withColumn(
    "prev_sale",
    lag("sale_amount").over(
        Window.partitionBy("product_name")
        .orderBy("sale_date"))
)

sales_product.filter(
    col("sale_amount") > col("prev_sale")
).show()

+----------+--------+-------+----------+-------------+-----------+---------+-------------------+------------------+-------+----------+-----------+------------+------------+--------+--------+-----------+----------+-------------------+---------+
|product_id|store_id|sale_id| sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|        store_name|   city|     state| store_type|manager_name|product_name|category|   brand|supplier_id|unit_price|data_quality_status|prev_sale|
+----------+--------+-------+----------+-------------+-----------+---------+-------------------+------------------+-------+----------+-----------+------------+------------+--------+--------+-----------+----------+-------------------+---------+
|      P107|    S104| SA1013|2026-02-01|            2|      16000|      UPI|              Valid|Metro Mart Chennai|Chennai|Tamil Nadu|Supermarket| Sneha Patel|       Watch| Fashion|Fastrack|       S206|      8000|              Valid|     8000|
+----------+--------+---

In [63]:
stores_df.createOrReplaceTempView("stores")
products_clean.createOrReplaceTempView("products")
inventory_clean.createOrReplaceTempView("inventory")
sales_clean.createOrReplaceTempView("sales")
suppliers_flat.createOrReplaceTempView("suppliers")

In [64]:
spark.sql("""
SELECT * FROM sales
""").show()

+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|   payment_m|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+------------+-------------------+----------------+-----+----+
| SA1001|    S101|      P101|2026-01-10|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1002|    S101|      P102|2026-01-10|            2|      50000|        Card|              Valid|    High Revenue|    1|2026|
| SA1003|    S102|      P101|2026-01-11|            1|      65000|         UPI|              Valid|    High Revenue|    1|2026|
| SA1004|    S103|      P106|2026-01-12|            4|      18000|        Cash|              Valid|  Medium Revenue|    1|2026|
| SA1005|    S104|      P107|2026-01-12|            1|       8000|        Card|              Valid|     

In [65]:
spark.sql("""
SELECT category,
COUNT(*) total_products
FROM products
GROUP BY category
""").show()

+-----------+--------------+
|   category|total_products|
+-----------+--------------+
|    Fashion|             4|
|Electronics|             5|
|  Furniture|             3|
+-----------+--------------+



In [66]:
spark.sql("""
SELECT st.store_name,
SUM(sa.sale_amount) revenue
FROM sales sa
JOIN stores st
ON sa.store_id=st.store_id
GROUP BY st.store_name
""").show()

+--------------------+-------+
|          store_name|revenue|
+--------------------+-------+
|Metro Mart Bangalore|  65000|
|    Metro Mart Kochi|  32000|
|   Metro Mart Jaipur|  10000|
|   Metro Mart Mumbai|  30000|
|     Metro Mart Pune|  38000|
|Metro Mart Hyderabad| 154000|
|    Metro Mart Delhi|  20000|
|  Metro Mart Chennai|  24000|
+--------------------+-------+



In [67]:
spark.sql("""
SELECT st.city,
SUM(sa.sale_amount) revenue
FROM sales sa
JOIN stores st
ON sa.store_id=st.store_id
GROUP BY st.city
""").show()

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|  65000|
|    Kochi|  32000|
|  Chennai|  24000|
|   Mumbai|  30000|
|     Pune|  38000|
|    Delhi|  20000|
|Hyderabad| 154000|
|   Jaipur|  10000|
+---------+-------+



In [68]:
spark.sql("""
SELECT *
FROM inventory
WHERE stock_quantity<=reorder_level
""").show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+
|       I1003|    S101|      P104|             3|            5| 2026-01-11|              Valid|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|              Valid|
|       I1008|    S104|      P107|             4|            5| 2026-01-15|              Valid|
|       I1010|    S106|      P109|             0|            6| 2026-01-16|      Missing Stock|
|       I1011|    S107|      P110|             1|            3| 2026-01-17|              Valid|
+------------+--------+----------+--------------+-------------+-----------+-------------------+



In [69]:
spark.sql("""
SELECT *
FROM sales
WHERE product_id NOT IN
(SELECT product_id FROM products)
""").show()

+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------------+-----+----+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_m|data_quality_status|revenue_category|month|year|
+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------------+-----+----+
| SA1009|    S108|      P120|2026-01-15|            2|      10000|     Cash|              Valid|     Low Revenue|    1|2026|
+-------+--------+----------+----------+-------------+-----------+---------+-------------------+----------------+-----+----+



In [70]:
spark.sql("""
SELECT *
FROM products
WHERE supplier_id NOT IN
(SELECT supplier_id FROM suppliers)
""").show()

+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+
|product_id|product_name|   category|    brand|supplier_id|unit_price|data_quality_status|price_category|
+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+
|      P107|       Watch|    Fashion| Fastrack|       S206|      8000|              Valid|        Budget|
|      P108|    Backpack|    Fashion|Wildcraft|       S206|      2500|              Valid|        Budget|
|      P111|  Headphones|Electronics|     Sony|       S999|      3000|              Valid|        Budget|
|      P112|     T-Shirt|    Fashion|     Puma|    UNKNOWN|      1500|            Invalid|        Budget|
+----------+------------+-----------+---------+-----------+----------+-------------------+--------------+



In [71]:
spark.sql("""
SELECT product_id,
SUM(sale_amount) revenue
FROM sales
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 5
""").show()

+----------+-------+
|product_id|revenue|
+----------+-------+
|      P101| 130000|
|      P102|  75000|
|      P109|  38000|
|      P110|  32000|
|      P107|  24000|
+----------+-------+



In [73]:
spark.sql("""
SELECT payment_m,
SUM(sale_amount) total_revenue
FROM sales
GROUP BY payment_m
""").show()

+------------+-------------+
|   payment_m|total_revenue|
+------------+-------------+
|        Card|       133000|
|        Cash|        35500|
|Not Provided|        14000|
|         UPI|       190500|
+------------+-------------+



In [74]:
sales_clean.write.mode("overwrite") \
    .parquet("gold/sales")

In [75]:
sales_clean.write.mode("overwrite") \
    .partitionBy("year", "month") \
    .parquet("gold/sales")

In [76]:
march_sales = [
("SA1016","S101","P102","2026-03-01",2,50000,"UPI"),
("SA1017","S104","P107","2026-03-02",1,8000,"Card"),
("SA1018","S105","P108","2026-03-03",4,10000,"Cash")
]

columns = ["sale_id","store_id","product_id","sale_date",
           "quantity_sold","sale_amount","payment_mode"]

march_df = spark.createDataFrame(march_sales,columns)

march_df.write.mode("overwrite") \
    .csv("incremental_sales_march.csv",header=True)

In [77]:
incremental_df = spark.read.csv(
    "incremental_sales_march.csv",
    header=True,
    inferSchema=True
)

In [78]:
incremental_df.write.mode("append") \
    .parquet("silver/sales")

In [79]:
updated_sales = spark.read.parquet("silver/sales")

updated_product_revenue = updated_sales.groupBy(
    "product_id"
).agg(
    sum("sale_amount").alias("total_revenue")
)

updated_product_revenue.show()

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|      P110|        32000|
|      P105|        12000|
|      P102|       125000|
|      P106|        18000|
|      P107|        32000|
|      P120|        10000|
|      P103|            0|
|      P109|        38000|
|      P104|        14000|
|      P101|       130000|
|      P108|        30000|
+----------+-------------+



In [80]:
updated_store_revenue = updated_sales.groupBy(
    "store_id"
).agg(
    sum("sale_amount").alias("total_revenue")
)

updated_store_revenue.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S105|        30000|
|    S102|        65000|
|    S106|        38000|
|    S104|        32000|
|    S107|        32000|
|    S101|       204000|
|    S108|        10000|
|    S103|        30000|
+--------+-------------+



In [81]:
updated_sales = updated_sales.withColumn(
    "month",month("sale_date")
).withColumn(
    "year",year("sale_date")
)

updated_sales.write.mode("overwrite") \
    .partitionBy("year","month") \
    .parquet("gold/sales")

In [82]:
before_count = sales_clean.count()

after_count = updated_sales.count()

print("Before Incremental Load :", before_count)
print("After Incremental Load :", after_count)

Before Incremental Load : 15
After Incremental Load : 18


In [83]:
store_report = retail_df.groupBy(
    "store_id",
    "store_name",
    "city",
    "state"
).agg(
    count("sale_id").alias("total_sales"),
    sum("sale_amount").alias("total_revenue")
)

store_report.show()

+--------+--------------------+---------+-----------+-----------+-------------+
|store_id|          store_name|     city|      state|total_sales|total_revenue|
+--------+--------------------+---------+-----------+-----------+-------------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|          1|        38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|          1|        32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|          4|       154000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|          2|        30000|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|          2|        20000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|          2|        65000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|          2|        24000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|          1|        10000|
+--------+--------------------+---------+-----------+-----------+-------------+



In [84]:
product_report = retail_df.groupBy(
    "product_id",
    "product_name",
    "category",
    "brand"
).agg(
    sum("quantity_sold").alias("total_quantity_sold"),
    sum("sale_amount").alias("total_revenue")
)

product_report.show()

+----------+------------+-----------+------------+-------------------+-------------+
|product_id|product_name|   category|       brand|total_quantity_sold|total_revenue|
+----------+------------+-----------+------------+-------------------+-------------+
|      P110|        Sofa|  Furniture|      Godrej|                  1|        32000|
|      P104|Office Chair|  Furniture| Featherlite|                  2|        14000|
|      P101|      Laptop|Electronics|      Lenovo|                  2|       130000|
|      P109|Refrigerator|Electronics|   Whirlpool|                  1|        38000|
|      P105| Study Table|  Furniture|Urban Ladder|                  1|        12000|
|      P107|       Watch|    Fashion|    Fastrack|                  3|        24000|
|      P102|      Mobile|Electronics|     Samsung|                  3|        75000|
|      P108|    Backpack|    Fashion|   Wildcraft|                  8|        20000|
|      P106|       Shoes|    Fashion|        Nike|               

In [85]:
inventory_report = inventory_product.select(
    "store_id",
    "product_id",
    "product_name",
    "stock_quantity",
    "reorder_level",
    "stock_status"
)

inventory_report.show()

+--------+----------+------------+--------------+-------------+----------------+
|store_id|product_id|product_name|stock_quantity|reorder_level|    stock_status|
+--------+----------+------------+--------------+-------------+----------------+
|    S101|      P101|      Laptop|            10|            5|Sufficient Stock|
|    S101|      P102|      Mobile|            25|           10|Sufficient Stock|
|    S101|      P104|Office Chair|             3|            5|Reorder Required|
|    S102|      P101|      Laptop|             8|            5|Sufficient Stock|
|    S102|      P103|  Television|             5|            4|Sufficient Stock|
|    S103|      P105| Study Table|             2|            5|Reorder Required|
|    S103|      P106|       Shoes|            30|           10|Sufficient Stock|
|    S104|      P107|       Watch|             4|            5|Reorder Required|
|    S105|      P108|    Backpack|            50|           20|Sufficient Stock|
|    S106|      P109|Refrige

In [86]:
supplier_report = suppliers_flat.select(
    "supplier_id",
    "supplier_name",
    "city",
    "rating",
    "supplier_quality",
    "phone",
    "email"
)

supplier_report.show()

+-----------+--------------------+---------+------+----------------+------------+--------------------+
|supplier_id|       supplier_name|     city|rating|supplier_quality|       phone|               email|
+-----------+--------------------+---------+------+----------------+------------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|       Excellent|  9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|            Good|Not Provided|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|            Good|  9876500013|        Not Provided|
|       S204|  Urban Furniture Co|    Delhi|   4.0|            Good|  9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|         Average|Not Provided|        Not Provided|
+-----------+--------------------+---------+------+----------------+------------+--------------------+



In [92]:
from pyspark.sql.functions import countDistinct, sum

In [93]:
category_report = retail_df.groupBy(
    "category"
).agg(
    countDistinct("product_id").alias("total_products"),
    sum("quantity_sold").alias("total_quantity_sold"),
    sum("sale_amount").alias("total_revenue")
)

category_report.show()

+-----------+--------------+-------------------+-------------+
|   category|total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|    Fashion|             3|                 15|        62000|
|       NULL|             1|                  2|        10000|
|Electronics|             4|                  7|       243000|
|  Furniture|             3|                  4|        58000|
+-----------+--------------+-------------------+-------------+



In [95]:
payment_report = sales_clean.groupBy(
    "payment_m"
).agg(
    count("*").alias("total_transactions"),
    sum("sale_amount").alias("total_revenue")
)

payment_report.show()

+------------+------------------+-------------+
|   payment_m|total_transactions|total_revenue|
+------------+------------------+-------------+
|        Card|                 5|       133000|
|        Cash|                 3|        35500|
|Not Provided|                 1|        14000|
|         UPI|                 6|       190500|
+------------+------------------+-------------+

